This notebook compares the CFD simulation and the experimental results.

In [ ]:
import logging
from pathlib import Path
from typing import List
import os
import json
import sqlite3

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from omegaconf import OmegaConf


# set up logging
logging.basicConfig(level=logging.INFO)

# set up matplotlib inline for Jupyter notebooks
try:
    from IPython import get_ipython

    get_ipython().run_line_magic("config", "InlineBackend.figure_formats = ['svg']")
except ImportError:
    logging.warning("IPython not found.")

from utils import energetics_calculator, calc_dist, get_bout_type, get_bout

# load the folder path from the json file
with open(Path.cwd() / "folder_path.json", "r") as f:
    folder_path = json.load(f)
root_folder = Path(folder_path["main"])

# Path to the SQLite database
db_file = root_folder / "results.db"
assert db_file.is_file(), f"Database file {db_file} does not exist"

# use custom matplotlib style
plt.style.use(root_folder / "main.mplstyle")
cfg = OmegaConf.load(root_folder / "config.yaml")
FPS: int = cfg.FPS

# Load simulation result

In [ ]:
out_folder = Path(os.environ["OUTPUT_FOLDER"])

# derive age_str, fish_id, t0 and t1 from the out_folder path
rel_path = out_folder.relative_to(root_folder)
age_str, _, fish_id, frame_name = rel_path.parts
_, t0_str, t1_str = frame_name.split("_")
t0, t1 = int(t0_str), int(t1_str)

logging.info(f"age: {age_str}, fish_id: {fish_id}, t0: {t0}, t1: {t1}")

In [ ]:
power_fname = out_folder / "Eel3d" / "Eel3d_Power_spent_struct_no_0"
en_sim = energetics_calculator(power_fname)
logging.info(f"Energy spent in simulation: {en_sim:.4f} erg")

# alternatively, we can calculate the distance as the norm of the last and first time frame
dist_sim, res = calc_dist(out_folder, return_path=True)
logging.info(f"Distance traveled in simulation: {dist_sim:.4f} cm")
logging.info(f"{dist_sim / en_sim} cm/erg")

# Compare with the recording

In [ ]:
with np.load(out_folder / "tmp.npz") as f:
    coords_raw = f["coords_raw"]
    t_list = f["t_list"]
    t_all = f["t_all"]
    speed_all = f["speed_all"]

In [ ]:
dist_sim = np.linalg.norm(coords_raw[-1, 4] - coords_raw[0, 4])
logging.info(f"Distance traveled in the experiment: {dist_sim:.4f} cm")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
fig.suptitle("Movement trajectory of tip of the fish head")
ax1.set_title("CFD")
ax1.scatter(res[:, 0], res[:, 1], color="black", alpha=0.05, s=1)
ax1.set_aspect("equal")
ax1.scatter(res[0, 0], res[0, 1], color="red", s=20, marker="*")

ax2.set_title("tracking")
idx = 4
ax2.scatter(*(coords_raw[:, idx]).T, color="black", alpha=0.5, s=1)
ax2.set_aspect("equal")
ax2.scatter(*coords_raw[0, idx], color="red", s=20, marker="*")

# turn off all axes
# ax1.axis("off")
# ax2.axis("off")
# add a star to the first point

In [ ]:
assert t_list[0] == 0, "Time should start at 0"

# Load the simulation time array
t_arr = np.loadtxt(power_fname, usecols=0)

In [ ]:
# get the time of the last frame frome the simulation (which may not have finished)
end_time: float = t_arr[-1]

idx = np.searchsorted(t_list, end_time)
assert idx <= t1 - t0, "idx should be less than or equal to t1 - t0"
t_ds_clip = t_list[:idx] + t_all[t0]
logging.info(f"simulation: {t_ds_clip[0]:.2f}--{t_ds_clip[-1]:.2f}s")
v2d_ds = speed_all[t0 : t0 + idx]

In [ ]:
distances = np.linalg.norm(np.diff(res, axis=0), axis=1)
speed = distances * FPS

# calculate moving average of the speed using a window of 20 points
window_size = 50  # moving average window size
df_speed = pd.DataFrame(speed, columns=["speed"])
df_speed["t"] = t_arr[1:] + t_all[t0]
df_speed_avg = df_speed.rolling(window=window_size).mean()
idx = np.searchsorted(df_speed["t"], t_ds_clip)
df_speed_avg_downsampled = df_speed_avg.iloc[idx]
df_speed_avg_downsampled

In [ ]:
fig, ax = plt.subplots(figsize=(2, 1))

bout_type = get_bout_type(cfg, age_str, fish_id, t0, t1)
burn_in = 0
if bout_type == "cont":
    burn_in = 0.5
    if out_folder.parent.name == "02182025133316" and out_folder.name == "frame_200_1200":
        logging.info("There was a turn bout before the continuous swimming")
        burn_in = 1.5
    logging.info(f"Continuous bout type, set burn_in to {burn_in} seconds")

idx = t_ds_clip >= (burn_in + t_ds_clip[0])
start_time = t_ds_clip[idx][0]
ax.plot((t_ds_clip[idx] - start_time) * 1000, v2d_ds[idx], label="experiment", color="#7c7c7c", linewidth=1)
ax.plot(
    (df_speed_avg_downsampled["t"][idx] - start_time) * 1000,
    df_speed_avg_downsampled["speed"][idx],
    label="simulation",
    color="black",
    linewidth=1,
)
# ax.legend()
# # set alpha in the elements in legend to 1
# for legline in ax.get_legend().get_lines():
#     legline.set_alpha(1.0)
ax.set(xlabel="time (ms)", ylabel="v(cm/s)")
# remove patches
[x.patch.set_visible(False) for x in (fig, ax)]
fig.savefig(out_folder / "velocity_comp.svg", transparent=True)

In [ ]:
len(speed_all), len(t_all)

In [ ]:
bout = get_bout(cfg, age_str, fish_id, t0, t1)

RE_res = []
# hard-coded FPS value for the experiment
FPS_exp: int = 250
if bout_type == "cont":
    # for continuous bouts, we calculate the mean speed over the entire
    # sequence of bouts with a burn-in period such that the fish reaches a steady state
    s, e = bout.start_frame + int(burn_in * FPS_exp), bout.end_frame
    mean_speed_exp = np.nanmean(speed_all[s:e])
    time_str = f"t >= {t_all[s]} and t <= {t_all[e]}"
    logging.info(
        f"Calculating mean speed for bout {s}-{e} with time condition: {time_str}"
    )
    df_tmp = df_speed_avg.query(time_str)
    mean_speed_sim = df_tmp["speed"].mean()
    RE = np.abs(mean_speed_sim - mean_speed_exp) / mean_speed_exp * 100
    # calculate the distance traveled in the experiment and simulation (integral of speed)
    dist_sim = np.trapezoid(df_tmp["speed"], df_tmp["t"])
    dist_exp = mean_speed_exp * (t_all[e] - t_all[s])
    # calculate the energy spent in the experiment and simulation with the same time range in the simulation
    en_sim = energetics_calculator(power_fname, start_frame=df_tmp.index[0], end_frame=df_tmp.index[-1])
    RE_res.append(
        (age_str, fish_id, s, e, mean_speed_exp, mean_speed_sim, RE, dist_exp, dist_sim, en_sim, bout_type)
    )
elif bout_type == "burst":
    # for burst-and-coast bouts, they are simulated together
    # but here we calculate the mean speed for each bout separately
    start_frame_each: List[int] = bout.get("start_frame_each", [])
    frame_idx = [bout.start_frame]
    frame_idx.extend(start_frame_each)
    frame_idx.append(bout.end_frame)

    for s, e in zip(frame_idx[:-1], frame_idx[1:]):
        mean_speed_exp = np.nanmean(speed_all[s:e])
        time_str = f"t >= {t_all[s]} and t <= {t_all[e]}"
        logging.info(
            f"Calculating mean speed for bout {s}-{e} with time condition: {time_str}"
        )
        df_tmp = df_speed_avg.query(time_str)
        mean_speed_sim = df_tmp["speed"].mean()
        RE = np.abs(mean_speed_sim - mean_speed_exp) / mean_speed_exp * 100
        # calculate the distance traveled in the experiment and simulation (integral of speed)
        dist_sim = np.trapezoid(df_tmp["speed"], df_tmp["t"])
        dist_exp = mean_speed_exp * (t_all[e] - t_all[s])
        # calculate the energy spent in the experiment and simulation with the same time range in the simulation
        en_sim = energetics_calculator(power_fname, start_frame=df_tmp.index[0], end_frame=df_tmp.index[-1])
        RE_res.append(
            (age_str, fish_id, s, e, mean_speed_exp, mean_speed_sim, RE, dist_exp, dist_sim, en_sim, bout_type)
        )
else:
    raise ValueError(f"Unknown bout type: {bout_type}")

logging.info(f"{RE=}")

In [ ]:
# Connect to the database
conn = sqlite3.connect(db_file)

# Insert each result tuple into the 'results' table
insert_sql = '''
INSERT INTO results (
    dpf,
    fish,
    start_frame,
    end_frame,
    exp_speed,
    cfd_speed,
    "RE (%)",
    exp_distance,
    cfd_distance,
    ibamr_energy,
    bout_type
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
'''
with conn:
    for row in RE_res:
        conn.execute(insert_sql, row)

conn.close()

logging.info(f"Results saved to {db_file}")

In [ ]:
# # read the results back from the database
# with sqlite3.connect(db_file) as conn:
#     df_tmp = pd.read_sql_query("SELECT * FROM results", conn)

# df_tmp

# Alternative definition of distance travelled

In [ ]:
# integrate the speed to get the distance traveled
dist_sim = np.trapezoid(
    df_speed_avg["speed"][window_size - 1 :], x=df_speed_avg["t"][window_size - 1 :]
)
logging.info(f"Distance traveled in simulation: {dist_sim:.4f} cm")

In [ ]:
dist_rec = np.linalg.norm(coords_raw[0, 4] - coords_raw[-1, 4])
logging.info(f"Distance traveled in the recording: {dist_rec:.2f} cm")